# 01 — Dataset V2 Preparation for KM Thermal Cold-Region Segmentation

This notebook creates the **clean Dataset V2** from raw FIR thermal images and sorted pixel-label masks.

Main assumptions used here:

1. FIR images and masks are matched by **natural sorted order**.
2. Subject, doctor/group token, and repeat ID are inherited from the paired FIR image filename, e.g. `S07-A-2.bmp`.
3. Mask labels are mapped as: `0/1 = background`, `2 = body/abdomen`, `3 = cold region`.
4. Body mask includes cold pixels: `body = label 2 OR label 3`.
5. Masks larger than FIR images, especially `260×320` masks for `256×320` images, are center-cropped to image size.

Run top to bottom before training any model.


In [ ]:
# ============================================================
# SECTION 00 — CONFIGURATION
# ============================================================

from pathlib import Path
import re
import json
import shutil
import hashlib
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")

ROOT = Path("/media/data/rPPG/rPPG_Data/SHARE/PixelLabelData")
IMG_DIR = ROOT / "FIR_input"
MASK_DIR = ROOT / "PixelLabelData"

OUT_ROOT = ROOT / "dataset_v2_prepared"

IMAGE_EXTS = [".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]
MASK_EXTS = [".png", ".bmp", ".jpg", ".jpeg", ".tif", ".tiff"]

KNOWN_EMPTY_OR_CORRUPT_SAMPLE_IDS = {"S08-A-3"}
EXPECTED_DOCTOR_GROUPS = ["A", "B", "C", "D"]
EXPECTED_REPEAT_IDS = [1, 2, 3]

FORCE_REBUILD = True
SAVE_ALL_OVERLAYS = True

print("ROOT:", ROOT)
print("IMG_DIR exists:", IMG_DIR.exists(), IMG_DIR)
print("MASK_DIR exists:", MASK_DIR.exists(), MASK_DIR)
print("OUT_ROOT:", OUT_ROOT)


In [ ]:
# ============================================================
# SECTION 01 — OUTPUT FOLDERS
# ============================================================

DIRS = {
    "images_png": OUT_ROOT / "images_png",
    "images_npy": OUT_ROOT / "images_npy",
    "masks_original_aligned": OUT_ROOT / "masks_original_aligned",
    "body_masks": OUT_ROOT / "body_masks",
    "cold_masks": OUT_ROOT / "cold_masks",
    "overlay_qc": OUT_ROOT / "overlay_qc",
    "figures": OUT_ROOT / "figures",
    "tables": OUT_ROOT / "tables",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Created folders under:", OUT_ROOT)


In [ ]:
# ============================================================
# SECTION 02 — HELPER FUNCTIONS
# ============================================================

def natural_sort_key(x):
    x = str(x)
    parts = re.split(r"(\d+)", x)
    return [int(p) if p.isdigit() else p.lower() for p in parts]


def natural_sort_string(x):
    x = str(x)
    parts = re.split(r"(\d+)", x)
    out = []
    for p in parts:
        if p.isdigit():
            out.append(p.zfill(8))
        else:
            out.append(p.lower())
    return "_".join(out)


def sort_paths(paths):
    return sorted([Path(p) for p in paths], key=lambda p: natural_sort_key(p.name))


def list_files(folder, exts):
    folder = Path(folder)
    files = []
    for ext in exts:
        files.extend(folder.rglob(f"*{ext}"))
        files.extend(folder.rglob(f"*{ext.upper()}"))
    files = sorted(set(files), key=lambda p: natural_sort_key(p.name))
    return files


def parse_image_metadata(image_name):
    stem = Path(str(image_name)).stem
    m = re.search(r"(S\d+)[-_]?([A-D])[-_ ]?(\d+)", stem, flags=re.IGNORECASE)
    if m:
        subject_id = m.group(1).upper()
        doctor_group = m.group(2).upper()
        repeat_id = int(m.group(3))
        sample_id = f"{subject_id}-{doctor_group}-{repeat_id}"
        return subject_id, doctor_group, repeat_id, sample_id, "parsed_from_image_filename"

    m2 = re.search(r"(S\d+)", stem, flags=re.IGNORECASE)
    subject_id = m2.group(1).upper() if m2 else "UNK"
    return subject_id, "UNK", -1, stem, "metadata_incomplete"


def load_gray(path):
    arr = np.array(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def save_gray_image(path, arr):
    arr = np.asarray(arr)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if arr.dtype == np.bool_:
        arr = arr.astype(np.uint8) * 255
    elif arr.dtype not in [np.uint8, np.uint16, np.int32, np.int16]:
        arr = arr.astype(np.float32)
        lo, hi = np.nanmin(arr), np.nanmax(arr)
        arr = ((arr - lo) / (hi - lo + 1e-8) * 255).clip(0, 255).astype(np.uint8)

    Image.fromarray(arr).save(path)


def normalize01(x):
    x = np.asarray(x).astype(np.float32)
    lo, hi = np.percentile(x, [1, 99])
    x = np.clip(x, lo, hi)
    return (x - lo) / (hi - lo + 1e-8)


def align_mask_to_image(mask, image_shape):
    ih, iw = image_shape[:2]
    mh, mw = mask.shape[:2]

    if (mh, mw) == (ih, iw):
        return mask, "same_shape"

    if mh >= ih and mw >= iw:
        top = max((mh - ih) // 2, 0)
        left = max((mw - iw) // 2, 0)
        cropped = mask[top:top + ih, left:left + iw]
        if cropped.shape[:2] == (ih, iw):
            return cropped, f"center_crop_from_{mh}x{mw}_to_{ih}x{iw}_top{top}_left{left}"

    pil = Image.fromarray(mask.astype(np.uint8))
    pil = pil.resize((iw, ih), resample=Image.NEAREST)
    resized = np.array(pil)
    return resized, f"nearest_resize_from_{mh}x{mw}_to_{ih}x{iw}"


def make_overlay(img, label_mask, alpha_body=0.20, alpha_cold=0.58):
    img01 = normalize01(img)
    rgb = np.stack([img01, img01, img01], axis=-1)

    body = (label_mask == 2) | (label_mask == 3)
    cold = label_mask == 3

    rgb[body, 0] = (1 - alpha_body) * rgb[body, 0] + alpha_body * 1.0
    rgb[body, 1] = (1 - alpha_body) * rgb[body, 1] + alpha_body * 0.78
    rgb[body, 2] = (1 - alpha_body) * rgb[body, 2] + alpha_body * 0.05

    rgb[cold, 0] = (1 - alpha_cold) * rgb[cold, 0] + alpha_cold * 1.0
    rgb[cold, 1] = (1 - alpha_cold) * rgb[cold, 1] + alpha_cold * 0.05
    rgb[cold, 2] = (1 - alpha_cold) * rgb[cold, 2] + alpha_cold * 0.05

    return np.clip(rgb, 0, 1)


def file_md5(path, max_bytes=None):
    h = hashlib.md5()
    with open(path, "rb") as f:
        if max_bytes is None:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        else:
            h.update(f.read(max_bytes))
    return h.hexdigest()


def label_values_string(mask):
    vals = np.unique(mask)
    return "|".join(map(str, vals.tolist()))


def problem_flags_for_row(sample_id, label_values, body_pixels, cold_pixels, cold_body_ratio, shape_mismatch, metadata_ok):
    flags = []
    allowed = {0, 1, 2, 3}
    observed = set(map(int, label_values))

    if not observed.issubset(allowed):
        flags.append("unexpected_label_values")
    if shape_mismatch:
        flags.append("shape_mismatch_corrected")
    if not metadata_ok:
        flags.append("metadata_parse_failed")
    if body_pixels == 0:
        flags.append("empty_body")
    if cold_pixels == 0:
        flags.append("empty_cold")
    if body_pixels > 0 and cold_body_ratio > 0.50:
        flags.append("very_large_cold_fraction")
    if body_pixels > 0 and 0 < cold_pixels < 50:
        flags.append("tiny_cold_region")
    if sample_id in KNOWN_EMPTY_OR_CORRUPT_SAMPLE_IDS:
        flags.append("known_flag_S08_A_3")

    return ";".join(flags) if flags else "OK"

print("Helpers ready.")


In [ ]:
# ============================================================
# SECTION 03 — BUILD SORTED IMAGE-MASK PAIRING
# ============================================================

image_files = list_files(IMG_DIR, IMAGE_EXTS) if IMG_DIR.exists() else []
mask_files = list_files(MASK_DIR, MASK_EXTS) if MASK_DIR.exists() else []

if len(image_files) == 0 or len(mask_files) == 0:
    raise FileNotFoundError(
        "Could not find raw FIR images or mask files. Check ROOT, IMG_DIR, and MASK_DIR in Section 00."
    )

print("Images found:", len(image_files))
print("Masks found :", len(mask_files))

if len(image_files) != len(mask_files):
    print("WARNING: image and mask counts differ.")
    print("The notebook will pair only min(counts) by sorted order.")

n_pairs = min(len(image_files), len(mask_files))
image_files = image_files[:n_pairs]
mask_files = mask_files[:n_pairs]

rows = []
for i, (img_path, mask_path) in enumerate(zip(image_files, mask_files)):
    subject_id, doctor_group, repeat_id, sample_id, parse_source = parse_image_metadata(img_path.name)
    rows.append({
        "pair_index": i,
        "sample_id": sample_id,
        "subject_id": subject_id,
        "doctor_group": doctor_group,
        "repeat_id": repeat_id,
        "metadata_parse_source": parse_source,
        "image_name": img_path.name,
        "mask_name": mask_path.name,
        "image_path": str(img_path),
        "mask_path": str(mask_path),
        "sorted_pairing_assumption": "image_i_matches_mask_i_after_natural_sort",
    })

pair_df = pd.DataFrame(rows)
pair_df.to_csv(DIRS["tables"] / "00_sorted_pairing_manifest_raw.csv", index=False)

print(pair_df.shape)
display(pair_df.head(12))


In [ ]:
# ============================================================
# SECTION 04 — EXPECTED GRID AND MISSING SAMPLE AUDIT
# ============================================================

subjects = sorted(pair_df["subject_id"].dropna().unique(), key=natural_sort_key)
subjects = [s for s in subjects if s != "UNK"]

expected_rows = []
for subject_id, doctor_group, repeat_id in product(subjects, EXPECTED_DOCTOR_GROUPS, EXPECTED_REPEAT_IDS):
    sample_id = f"{subject_id}-{doctor_group}-{repeat_id}"
    expected_rows.append({
        "subject_id": subject_id,
        "doctor_group": doctor_group,
        "repeat_id": repeat_id,
        "sample_id": sample_id,
    })

expected_df = pd.DataFrame(expected_rows)
observed_set = set(pair_df["sample_id"].astype(str))
expected_df["available"] = expected_df["sample_id"].isin(observed_set)
missing_df = expected_df[~expected_df["available"]].copy().reset_index(drop=True)

summary_grid = (
    pair_df.groupby(["subject_id", "doctor_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=subjects, columns=EXPECTED_DOCTOR_GROUPS, fill_value=0)
)

expected_df.to_csv(DIRS["tables"] / "01_expected_subject_doctor_repeat_grid.csv", index=False)
missing_df.to_csv(DIRS["tables"] / "02_missing_expected_samples.csv", index=False)
summary_grid.to_csv(DIRS["tables"] / "03_subject_doctor_group_counts.csv")

print("Expected samples:", len(expected_df))
print("Observed samples :", len(pair_df))
print("Missing expected :", len(missing_df))
display(summary_grid)
display(missing_df)


In [ ]:
# ============================================================
# SECTION 05 — PREPARE ALIGNED DATASET V2
# ============================================================

prepared_rows = []

for _, row in pair_df.iterrows():
    img_path = Path(row["image_path"])
    mask_path = Path(row["mask_path"])

    img = load_gray(img_path)
    mask_orig = load_gray(mask_path)
    mask_aligned, alignment_method = align_mask_to_image(mask_orig, img.shape)

    mask_aligned = mask_aligned.astype(np.uint8)
    label_values = np.unique(mask_aligned)

    body_mask = ((mask_aligned == 2) | (mask_aligned == 3)).astype(np.uint8)
    cold_mask = (mask_aligned == 3).astype(np.uint8)

    sample_id = row["sample_id"]
    safe_id = str(sample_id).replace("/", "_")

    prepared_image_png = DIRS["images_png"] / f"{safe_id}.png"
    prepared_image_npy = DIRS["images_npy"] / f"{safe_id}.npy"
    aligned_label_path = DIRS["masks_original_aligned"] / f"{safe_id}_label.png"
    body_mask_path = DIRS["body_masks"] / f"{safe_id}_body.png"
    cold_mask_path = DIRS["cold_masks"] / f"{safe_id}_cold.png"
    overlay_path = DIRS["overlay_qc"] / f"{safe_id}_overlay.png"

    np.save(prepared_image_npy, img)
    save_gray_image(prepared_image_png, img)
    save_gray_image(aligned_label_path, mask_aligned)
    save_gray_image(body_mask_path, body_mask * 255)
    save_gray_image(cold_mask_path, cold_mask * 255)

    if SAVE_ALL_OVERLAYS:
        overlay = make_overlay(img, mask_aligned)
        Image.fromarray((overlay * 255).astype(np.uint8)).save(overlay_path)

    body_pixels = int(body_mask.sum())
    cold_pixels = int(cold_mask.sum())
    cold_body_ratio = float(cold_pixels / (body_pixels + 1e-8))

    shape_mismatch = tuple(mask_orig.shape[:2]) != tuple(img.shape[:2])
    metadata_ok = row["doctor_group"] != "UNK" and int(row["repeat_id"]) > 0 and row["subject_id"] != "UNK"

    flags = problem_flags_for_row(
        sample_id=sample_id,
        label_values=label_values,
        body_pixels=body_pixels,
        cold_pixels=cold_pixels,
        cold_body_ratio=cold_body_ratio,
        shape_mismatch=shape_mismatch,
        metadata_ok=metadata_ok,
    )

    use_for_training = True
    if "empty_body" in flags:
        use_for_training = False
    if "unexpected_label_values" in flags:
        use_for_training = False
    if "metadata_parse_failed" in flags:
        use_for_training = False
    if sample_id in KNOWN_EMPTY_OR_CORRUPT_SAMPLE_IDS:
        use_for_training = False

    prepared_rows.append({
        **row.to_dict(),
        "prepared_image_png": str(prepared_image_png),
        "prepared_image_npy": str(prepared_image_npy),
        "aligned_label_mask_path": str(aligned_label_path),
        "body_mask_path": str(body_mask_path),
        "cold_mask_path": str(cold_mask_path),
        "overlay_qc_path": str(overlay_path),
        "image_height": int(img.shape[0]),
        "image_width": int(img.shape[1]),
        "mask_original_height": int(mask_orig.shape[0]),
        "mask_original_width": int(mask_orig.shape[1]),
        "mask_aligned_height": int(mask_aligned.shape[0]),
        "mask_aligned_width": int(mask_aligned.shape[1]),
        "alignment_method": alignment_method,
        "shape_mismatch_corrected": bool(shape_mismatch),
        "label_values": label_values_string(mask_aligned),
        "body_pixels": body_pixels,
        "cold_pixels": cold_pixels,
        "cold_body_ratio": cold_body_ratio,
        "problem_flags": flags,
        "use_for_training": bool(use_for_training),
        "image_md5": file_md5(img_path),
        "mask_md5": file_md5(mask_path),
    })

manifest_v2 = pd.DataFrame(prepared_rows)
manifest_v2.to_csv(OUT_ROOT / "manifest_v2.csv", index=False)
manifest_v2.to_csv(DIRS["tables"] / "manifest_v2.csv", index=False)

problem_cases_v2 = manifest_v2[manifest_v2["problem_flags"] != "OK"].copy()
problem_cases_v2.to_csv(OUT_ROOT / "problem_cases_v2.csv", index=False)
problem_cases_v2.to_csv(DIRS["tables"] / "problem_cases_v2.csv", index=False)

print("Saved manifest:", OUT_ROOT / "manifest_v2.csv")
print("Manifest shape:", manifest_v2.shape)
print("Training usable:", int(manifest_v2["use_for_training"].sum()), "/", len(manifest_v2))
print("Problem cases:", len(problem_cases_v2))
display(manifest_v2.head())
display(problem_cases_v2[["sample_id", "image_name", "mask_name", "problem_flags", "body_pixels", "cold_pixels", "cold_body_ratio", "use_for_training"]].head(30))


In [ ]:
# ============================================================
# SECTION 06 — DATASET SUMMARY TABLES
# ============================================================

subject_summary = (
    manifest_v2.groupby("subject_id")
    .agg(
        n_samples=("sample_id", "count"),
        n_train_usable=("use_for_training", "sum"),
        mean_body_pixels=("body_pixels", "mean"),
        median_body_pixels=("body_pixels", "median"),
        mean_cold_pixels=("cold_pixels", "mean"),
        median_cold_pixels=("cold_pixels", "median"),
        min_cold_pixels=("cold_pixels", "min"),
        max_cold_pixels=("cold_pixels", "max"),
        mean_cold_body_ratio=("cold_body_ratio", "mean"),
        n_shape_mismatch=("shape_mismatch_corrected", "sum"),
    )
    .reset_index()
)

annotation_group_summary = (
    manifest_v2.groupby("doctor_group")
    .agg(
        n_samples=("sample_id", "count"),
        mean_body_pixels=("body_pixels", "mean"),
        mean_cold_pixels=("cold_pixels", "mean"),
        median_cold_pixels=("cold_pixels", "median"),
        mean_cold_body_ratio=("cold_body_ratio", "mean"),
    )
    .reset_index()
)

label_summary = (
    manifest_v2.groupby("label_values")
    .size()
    .reset_index(name="n_masks")
    .sort_values("n_masks", ascending=False)
)

alignment_summary = (
    manifest_v2.groupby("alignment_method")
    .size()
    .reset_index(name="n_cases")
    .sort_values("n_cases", ascending=False)
)

summary = {
    "n_total_pairs": int(len(manifest_v2)),
    "n_training_usable": int(manifest_v2["use_for_training"].sum()),
    "n_subjects": int(manifest_v2["subject_id"].nunique()),
    "subjects": sorted(manifest_v2["subject_id"].unique().tolist(), key=natural_sort_key),
    "n_problem_cases": int((manifest_v2["problem_flags"] != "OK").sum()),
    "n_shape_mismatch_corrected": int(manifest_v2["shape_mismatch_corrected"].sum()),
    "n_empty_body": int(manifest_v2["problem_flags"].str.contains("empty_body", regex=False).sum()),
    "n_empty_cold": int(manifest_v2["problem_flags"].str.contains("empty_cold", regex=False).sum()),
    "mean_cold_pixels": float(manifest_v2["cold_pixels"].mean()),
    "median_cold_pixels": float(manifest_v2["cold_pixels"].median()),
    "min_cold_pixels": int(manifest_v2["cold_pixels"].min()),
    "max_cold_pixels": int(manifest_v2["cold_pixels"].max()),
}

subject_summary.to_csv(OUT_ROOT / "dataset_summary_v2_by_subject.csv", index=False)
annotation_group_summary.to_csv(OUT_ROOT / "dataset_summary_v2_by_doctor_group.csv", index=False)
label_summary.to_csv(OUT_ROOT / "label_summary_v2.csv", index=False)
alignment_summary.to_csv(OUT_ROOT / "alignment_summary_v2.csv", index=False)

with open(OUT_ROOT / "dataset_summary_v2.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
display(subject_summary)
display(annotation_group_summary)
display(label_summary)
display(alignment_summary)


In [ ]:
# ============================================================
# SECTION 07 — PLOTS FOR DATASET V2
# ============================================================

plt.figure(figsize=(8, 4))
manifest_v2.groupby("subject_id")["sample_id"].count().reindex(subjects).plot(kind="bar")
plt.ylabel("Number of samples")
plt.title("Dataset V2: samples per subject")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_01_samples_per_subject.png", dpi=250)
plt.show()

plt.figure(figsize=(8, 4))
subject_summary.set_index("subject_id").reindex(subjects)["mean_cold_pixels"].plot(kind="bar")
plt.ylabel("Mean cold pixels")
plt.title("Dataset V2: cold-region size varies across subjects")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_02_mean_cold_pixels_by_subject.png", dpi=250)
plt.show()

plt.figure(figsize=(8, 4))
manifest_v2.boxplot(column="cold_pixels", by="subject_id", grid=False, rot=45)
plt.suptitle("")
plt.title("Dataset V2: cold pixel distribution by subject")
plt.ylabel("Cold pixels")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_03_cold_pixels_boxplot_by_subject.png", dpi=250)
plt.show()

plt.figure(figsize=(8, 4))
manifest_v2.boxplot(column="cold_body_ratio", by="subject_id", grid=False, rot=45)
plt.suptitle("")
plt.title("Dataset V2: cold/body ratio distribution by subject")
plt.ylabel("Cold / body ratio")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_04_cold_body_ratio_by_subject.png", dpi=250)
plt.show()

plt.figure(figsize=(7, 4))
annotation_group_summary.set_index("doctor_group").reindex(EXPECTED_DOCTOR_GROUPS)["mean_cold_pixels"].plot(kind="bar")
plt.ylabel("Mean cold pixels")
plt.title("Dataset V2: annotation size by doctor/group token")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_05_mean_cold_pixels_by_doctor_group.png", dpi=250)
plt.show()


In [ ]:
# ============================================================
# SECTION 08 — SUBJECT × DOCTOR/GROUP AVAILABILITY HEATMAP
# ============================================================

count_grid = (
    manifest_v2.groupby(["subject_id", "doctor_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=subjects, columns=EXPECTED_DOCTOR_GROUPS, fill_value=0)
)

plt.figure(figsize=(7, 4.5))
plt.imshow(count_grid.values, aspect="auto")
plt.xticks(range(len(count_grid.columns)), count_grid.columns)
plt.yticks(range(len(count_grid.index)), count_grid.index)
plt.colorbar(label="Number of repeats")
plt.title("Dataset V2: available repeats per subject and doctor/group")
for i in range(count_grid.shape[0]):
    for j in range(count_grid.shape[1]):
        plt.text(j, i, str(int(count_grid.values[i, j])), ha="center", va="center")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "v2_06_subject_doctor_group_heatmap.png", dpi=250)
plt.show()


In [ ]:
# ============================================================
# SECTION 09 — OVERLAY CONTACT SHEETS
# ============================================================

def make_contact_sheet(df, save_path, title, max_cols=4):
    df = df.copy().reset_index(drop=True)
    n = len(df)
    if n == 0:
        return None

    cols = min(max_cols, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))

    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes[None, :]
    elif cols == 1:
        axes = axes[:, None]

    for ax in axes.ravel():
        ax.axis("off")

    for idx, (_, r) in enumerate(df.iterrows()):
        ax = axes.ravel()[idx]
        overlay = np.array(Image.open(r["overlay_qc_path"]))
        ax.imshow(overlay)
        ax.set_title(
            f"{r['sample_id']}\nCold={int(r['cold_pixels'])}, Ratio={r['cold_body_ratio']:.3f}",
            fontsize=10,
        )
        ax.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=220)
    plt.show()
    return save_path

for subject_id in subjects:
    sub = manifest_v2[manifest_v2["subject_id"] == subject_id].copy()
    sub = sub.sort_values(["doctor_group", "repeat_id"], key=lambda s: s.map(natural_sort_string) if s.dtype == object else s)
    save_path = DIRS["figures"] / f"v2_07_overlay_contact_sheet_{subject_id}.png"
    make_contact_sheet(sub, save_path, f"Dataset V2 overlay QC — Subject {subject_id}", max_cols=4)


In [ ]:
# ============================================================
# SECTION 10 — PROBLEM CASE CONTACT SHEET
# ============================================================

problem_vis = manifest_v2[manifest_v2["problem_flags"] != "OK"].copy()

priority_order = [
    "empty_body",
    "known_flag_S08_A_3",
    "very_large_cold_fraction",
    "tiny_cold_region",
    "shape_mismatch_corrected",
    "empty_cold",
]

def problem_priority(flags):
    for i, key in enumerate(priority_order):
        if key in flags:
            return i
    return 999

problem_vis["priority"] = problem_vis["problem_flags"].apply(problem_priority)
problem_vis = problem_vis.sort_values(["priority", "subject_id", "doctor_group", "repeat_id"]).head(12)

if len(problem_vis) > 0:
    save_path = DIRS["figures"] / "v2_08_problem_case_contact_sheet.png"
    make_contact_sheet(problem_vis, save_path, "Dataset V2 problem-case examples", max_cols=4)
else:
    print("No problem cases detected.")


In [ ]:
# ============================================================
# SECTION 11 — FINAL CHECKS FOR TRAINING
# ============================================================

required_cols = [
    "sample_id",
    "subject_id",
    "doctor_group",
    "repeat_id",
    "prepared_image_png",
    "prepared_image_npy",
    "aligned_label_mask_path",
    "body_mask_path",
    "cold_mask_path",
    "image_height",
    "image_width",
    "body_pixels",
    "cold_pixels",
    "cold_body_ratio",
    "use_for_training",
]

missing_cols = [c for c in required_cols if c not in manifest_v2.columns]
assert len(missing_cols) == 0, f"Missing required columns: {missing_cols}"

usable = manifest_v2[manifest_v2["use_for_training"]].copy()
assert usable["subject_id"].nunique() >= 2, "Need at least 2 subjects for subject-wise splitting."
assert (usable["image_height"] == usable["mask_aligned_height"]).all(), "Image/mask height mismatch remains."
assert (usable["image_width"] == usable["mask_aligned_width"]).all(), "Image/mask width mismatch remains."

print("Dataset V2 preparation completed.")
print("Output root:", OUT_ROOT)
print("Main manifest:", OUT_ROOT / "manifest_v2.csv")
print("Usable samples:", len(usable), "/", len(manifest_v2))
print("Subjects:", sorted(usable["subject_id"].unique(), key=natural_sort_key))

final_table = usable[[
    "sample_id", "subject_id", "doctor_group", "repeat_id",
    "prepared_image_npy", "body_mask_path", "cold_mask_path",
    "body_pixels", "cold_pixels", "cold_body_ratio", "use_for_training"
]].copy()
final_table.to_csv(OUT_ROOT / "training_manifest_v2.csv", index=False)

print("Training manifest:", OUT_ROOT / "training_manifest_v2.csv")
display(final_table.head(20))


## Next step

After this notebook runs successfully, use:

```text
manifest_v2.csv
training_manifest_v2.csv
```

for the LOSO split builder and model training notebooks.
